# The Simulation Object

In [ ]:
import hoomd
print(hoomd.version.version)
print(hoomd.__file__)

In [ ]:
cpu = hoomd.device.CPU()

In [ ]:
gpu = hoomd.device.GPU()

In [ ]:
simulation = hoomd.Simulation(device=cpu)

In [ ]:
simulation = hoomd.Simulation(device=gpu)

In [ ]:
print(simulation.operations.integrator)

In [ ]:
print(simulation.operations.updaters[:])

In [ ]:
print(simulation.operations.writers[:])

# Performing Hard Particle Monte Carlo Simulations

In [ ]:
mc = hoomd.hpmc.integrate.ConvexPolyhedron()

In [ ]:
mc.shape["octahedron"] = dict(
    vertices=[
        (-0.5, 0, 0),
        (0.5, 0, 0),
        (0, -0.5, 0),
        (0, 0.5, 0),
        (0, 0, -0.5),
        (0, 0, 0.5),
    ]
)

In [ ]:
mc.nselect = 2
mc.d["octahedron"] = 0.15
mc.a["octahedron"] = 0.2

In [ ]:
cpu = hoomd.device.CPU()
simulation = hoomd.Simulation(device=cpu, seed=1)

In [ ]:
simulation.operations.integrator = mc

# Initializing the System State

In [ ]:
import itertools
import math

import hoomd
import numpy

In [ ]:
m = 4
N_particles = 2 * m**3

In [ ]:
spacing = 1.2
K = math.ceil(N_particles ** (1 / 3))
L = K * spacing

In [ ]:
x = numpy.linspace(-L / 2, L / 2, K, endpoint=False)
position = list(itertools.product(x, repeat=3))
print(position[0:4])

In [ ]:
position = position[0:N_particles]

In [ ]:
orientation = [(1, 0, 0, 0)] * N_particles

In [ ]:
# Hidden tutorial helper cell

import os
import math
import warnings

import fresnel
import IPython
import packaging.version

device = fresnel.Device()
tracer = fresnel.tracer.Path(device=device, w=300, h=300)

FRESNEL_MIN_VERSION = packaging.version.parse("0.13.0")
FRESNEL_MAX_VERSION = packaging.version.parse("0.14.0")

def render(position, orientation, L):
    # Version guard (matches tutorial behavior)
    if (
        "version" not in dir(fresnel)
        or packaging.version.parse(fresnel.version.version) < FRESNEL_MIN_VERSION
        or packaging.version.parse(fresnel.version.version) >= FRESNEL_MAX_VERSION
    ):
        warnings.warn(f"Unsupported fresnel version {getattr(fresnel.version, 'version', 'unknown')} - expect errors.")

    vertices = [
        (-0.5, 0, 0),
        (0.5, 0, 0),
        (0, -0.5, 0),
        (0, 0.5, 0),
        (0, 0, -0.5),
        (0, 0, 0.5),
    ]
    poly_info = fresnel.util.convex_polyhedron_from_vertices(vertices)

    scene = fresnel.Scene(device)

    geometry = fresnel.geometry.ConvexPolyhedron(scene, poly_info, N=len(position))
    geometry.material = fresnel.material.Material(
        color=fresnel.color.linear([0.01, 0.74, 0.26]),
        roughness=0.5,
    )
    geometry.position[:] = position[:]
    geometry.orientation[:] = orientation[:]
    geometry.outline_width = 0.01

    fresnel.geometry.Box(scene, [L, L, L, 0, 0, 0], box_radius=0.02)

    scene.lights = [
        fresnel.light.Light(direction=(0, 0, 1), color=(0.8, 0.8, 0.8), theta=math.pi),
        fresnel.light.Light(direction=(1, 1, 1), color=(1.1, 1.1, 1.1), theta=math.pi / 3),
    ]

    scene.camera = fresnel.camera.Orthographic(
        position=(L * 2, L, L * 2),
        look_at=(0, 0, 0),
        up=(0, 1, 0),
        height=L * 1.4 + 1,
    )

    scene.background_color = (1, 1, 1)
    scene.background_alpha = 1

    samples = 2000
    if "CI" in os.environ:
        samples = 100

    return IPython.display.Image(tracer.sample(scene, samples=samples)._repr_png_())

In [ ]:
render(position, orientation, L)

## Writing the Configuration to the File System

In [ ]:
import gsd.hoomd

In [ ]:
frame = gsd.hoomd.Frame()
frame.particles.N = N_particles
frame.particles.position = position
frame.particles.orientation = orientation

In [ ]:
frame.particles.typeid = [0] * N_particles

In [ ]:
frame.particles.types = ["octahedron"]

In [ ]:
frame.configuration.box = [L, L, L, 0, 0, 0]

In [ ]:
with gsd.hoomd.open(name="lattice.gsd", mode="w") as f:
    f.append(frame)

## Initializing a Simulation

In [ ]:
cpu = hoomd.device.CPU()
simulation = hoomd.Simulation(device=cpu)

In [ ]:
simulation.create_state_from_gsd(filename="lattice.gsd")

# Randomizing the System

In [ ]:
import math

import hoomd

In [ ]:
cpu = hoomd.device.CPU()
simulation = hoomd.Simulation(device=cpu, seed=12)

mc = hoomd.hpmc.integrate.ConvexPolyhedron()
mc.shape["octahedron"] = dict(
    vertices=[
        (-0.5, 0, 0),
        (0.5, 0, 0),
        (0, -0.5, 0),
        (0, 0.5, 0),
        (0, 0, -0.5),
        (0, 0, 0.5),
    ]
)

simulation.operations.integrator = mc
simulation.create_state_from_gsd(filename="lattice.gsd")

In [ ]:
initial_snapshot = simulation.state.get_snapshot()

In [ ]:
simulation.run(10e3)

In [ ]:
mc.translate_moves

In [ ]:
print(mc.translate_moves)
trans_acceptance_ratio = mc.translate_moves[0] / sum(mc.translate_moves)
trans_acceptance_ratio

In [ ]:
print(mc.rotate_moves)
rot_acceptance_ratio = mc.rotate_moves[0] / sum(mc.rotate_moves)
rot_acceptance_ratio

In [ ]:
mc.overlaps

In [ ]:
# Fresnel helper for HOOMD snapshots

import math
import os
import warnings

import fresnel
import IPython
import packaging.version

device = fresnel.Device()
tracer = fresnel.tracer.Path(device=device, w=300, h=300)

FRESNEL_MIN_VERSION = packaging.version.parse("0.13.0")
FRESNEL_MAX_VERSION = packaging.version.parse("0.14.0")

def render(snapshot):
    if (
        "version" not in dir(fresnel)
        or packaging.version.parse(fresnel.version.version) < FRESNEL_MIN_VERSION
        or packaging.version.parse(fresnel.version.version) >= FRESNEL_MAX_VERSION
    ):
        warnings.warn(
            f"Unsupported fresnel version {fresnel.version.version} - expect errors."
        )

    L = snapshot.configuration.box[0]

    vertices = [
        (-0.5, 0, 0),
        (0.5, 0, 0),
        (0, -0.5, 0),
        (0, 0.5, 0),
        (0, 0, -0.5),
        (0, 0, 0.5),
    ]
    poly_info = fresnel.util.convex_polyhedron_from_vertices(vertices)

    scene = fresnel.Scene(device)

    geometry = fresnel.geometry.ConvexPolyhedron(
        scene, poly_info, N=snapshot.particles.N
    )

    geometry.material = fresnel.material.Material(
        color=fresnel.color.linear([0.01, 0.74, 0.26]),
        roughness=0.5,
    )

    geometry.position[:] = snapshot.particles.position[:]
    geometry.orientation[:] = snapshot.particles.orientation[:]
    geometry.outline_width = 0.01

    fresnel.geometry.Box(scene, snapshot.configuration.box, box_radius=0.02)

    scene.lights = [
        fresnel.light.Light(direction=(0, 0, 1), color=(0.8, 0.8, 0.8), theta=math.pi),
        fresnel.light.Light(
            direction=(1, 1, 1), color=(1.1, 1.1, 1.1), theta=math.pi / 3
        ),
    ]

    scene.camera = fresnel.camera.Orthographic(
        position=(L * 2, L, L * 2),
        look_at=(0, 0, 0),
        up=(0, 1, 0),
        height=L * 1.4 + 1,
    )

    scene.background_color = (1, 1, 1)
    scene.background_alpha = 1

    samples = 2000
    if "CI" in os.environ:
        samples = 100

    return IPython.display.Image(
        tracer.sample(scene, samples=samples)._repr_png_()
    )

In [ ]:
final_snapshot = simulation.state.get_snapshot()
render(final_snapshot)

In [ ]:
initial_snapshot.particles.position[0:4]

In [ ]:
final_snapshot.particles.position[0:4]

In [ ]:
initial_snapshot.particles.orientation[0:4]

In [ ]:
final_snapshot.particles.orientation[0:4]

In [ ]:
hoomd.write.GSD.write(state=simulation.state, mode="wb", filename="random.gsd")

# Compressing the System

In [ ]:
import math

import hoomd

In [ ]:
cpu = hoomd.device.CPU()
simulation = hoomd.Simulation(device=cpu, seed=12)
simulation.create_state_from_gsd(filename="random.gsd")

In [ ]:
a = math.sqrt(2) / 2
V_particle = 1 / 3 * math.sqrt(2) * a**3

In [ ]:
initial_volume_fraction = (
    simulation.state.N_particles * V_particle / simulation.state.box.volume
)
print(initial_volume_fraction)

In [ ]:
mc = hoomd.hpmc.integrate.ConvexPolyhedron()
mc.shape["octahedron"] = dict(
    vertices=[
        (-0.5, 0, 0),
        (0.5, 0, 0),
        (0, -0.5, 0),
        (0, 0.5, 0),
        (0, 0, -0.5),
        (0, 0, 0.5),
    ]
)
simulation.operations.integrator = mc

In [ ]:
initial_box = simulation.state.box
final_box = hoomd.Box.from_box(initial_box)
final_volume_fraction = 0.57
final_box.volume = simulation.state.N_particles * V_particle / final_volume_fraction
compress = hoomd.hpmc.update.QuickCompress(
    trigger=hoomd.trigger.Periodic(10), target_box=final_box
)

In [ ]:
simulation.operations.updaters.append(compress)

In [ ]:
periodic = hoomd.trigger.Periodic(10)
tune = hoomd.hpmc.tune.MoveSize.scale_solver(
    moves=["a", "d"],
    target=0.2,
    trigger=periodic,
    max_translation_move=0.2,
    max_rotation_move=0.2,
)
simulation.operations.tuners.append(tune)

In [ ]:
while not compress.complete and simulation.timestep < 1e6:
    simulation.run(1000)

In [ ]:
simulation.timestep

In [ ]:
if not compress.complete:
    message = "Compression failed to complete"
    raise RuntimeError(message)

In [ ]:
mc.a["octahedron"]

In [ ]:
mc.d["octahedron"]

In [ ]:
# Fresnel snapshot renderer (hidden tutorial helper)

import math
import os
import warnings

import fresnel
import IPython
import packaging.version

device = fresnel.Device()
tracer = fresnel.tracer.Path(device=device, w=300, h=300)

FRESNEL_MIN_VERSION = packaging.version.parse("0.13.0")
FRESNEL_MAX_VERSION = packaging.version.parse("0.14.0")


def render(snapshot):
    if (
        "version" not in dir(fresnel)
        or packaging.version.parse(fresnel.version.version) < FRESNEL_MIN_VERSION
        or packaging.version.parse(fresnel.version.version) >= FRESNEL_MAX_VERSION
    ):
        warnings.warn(
            f"Unsupported fresnel version {fresnel.version.version} - expect errors."
        )

    L = snapshot.configuration.box[0]

    vertices = [
        (-0.5, 0, 0),
        (0.5, 0, 0),
        (0, -0.5, 0),
        (0, 0.5, 0),
        (0, 0, -0.5),
        (0, 0, 0.5),
    ]

    poly_info = fresnel.util.convex_polyhedron_from_vertices(vertices)

    scene = fresnel.Scene(device)

    geometry = fresnel.geometry.ConvexPolyhedron(
        scene, poly_info, N=snapshot.particles.N
    )

    geometry.material = fresnel.material.Material(
        color=fresnel.color.linear([0.01, 0.74, 0.26]),
        roughness=0.5,
    )

    geometry.position[:] = snapshot.particles.position[:]
    geometry.orientation[:] = snapshot.particles.orientation[:]
    geometry.outline_width = 0.01

    fresnel.geometry.Box(scene, snapshot.configuration.box, box_radius=0.02)

    scene.lights = [
        fresnel.light.Light(direction=(0, 0, 1), color=(0.8, 0.8, 0.8), theta=math.pi),
        fresnel.light.Light(
            direction=(1, 1, 1), color=(1.1, 1.1, 1.1), theta=math.pi / 3
        ),
    ]

    scene.camera = fresnel.camera.Orthographic(
        position=(L * 2, L, L * 2),
        look_at=(0, 0, 0),
        up=(0, 1, 0),
        height=L * 1.4 + 1,
    )

    scene.background_alpha = 1
    scene.background_color = (1, 1, 1)

    samples = 2000
    if "CI" in os.environ:
        samples = 100

    return IPython.display.Image(
        tracer.sample(scene, samples=samples)._repr_png_()
    )

In [ ]:
render(simulation.state.get_snapshot())

In [ ]:
hoomd.write.GSD.write(state=simulation.state, mode="wb", filename="compressed.gsd")

# Equilibrating the System

In [ ]:
import math

import hoomd

In [ ]:
cpu = hoomd.device.CPU()
simulation = hoomd.Simulation(device=cpu, seed=12)
mc = hoomd.hpmc.integrate.ConvexPolyhedron()
mc.shape["octahedron"] = dict(
    vertices=[
        (-0.5, 0, 0),
        (0.5, 0, 0),
        (0, -0.5, 0),
        (0, 0.5, 0),
        (0, 0, -0.5),
        (0, 0, 0.5),
    ]
)
simulation.operations.integrator = mc

In [ ]:
simulation.create_state_from_gsd(filename="compressed.gsd")

In [ ]:
gsd_writer = hoomd.write.GSD(
    filename="trajectory.gsd", trigger=hoomd.trigger.Periodic(1000), mode="wb"
)
simulation.operations.writers.append(gsd_writer)

In [ ]:
tune = hoomd.hpmc.tune.MoveSize.scale_solver(
    moves=["a", "d"],
    target=0.2,
    trigger=hoomd.trigger.And(
        [hoomd.trigger.Periodic(100), hoomd.trigger.Before(simulation.timestep + 5000)]
    ),
)
simulation.operations.tuners.append(tune)

In [ ]:
simulation.run(5000)

In [ ]:
simulation.run(100)

In [ ]:
rotate_moves = mc.rotate_moves
mc.rotate_moves[0] / sum(mc.rotate_moves)

In [ ]:
translate_moves = mc.translate_moves
mc.translate_moves[0] / sum(mc.translate_moves)

In [ ]:
simulation.run(2e4)

In [ ]:
render(simulation.state.get_snapshot())

In [ ]:
simulation.run(2e3)

In [ ]:
render(simulation.state.get_snapshot())

# Analyzing Trajectories

In [ ]:
import math

import freud
import gsd.hoomd
import matplotlib

%matplotlib inline
matplotlib.style.use("ggplot")
import matplotlib_inline

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

In [ ]:
traj = gsd.hoomd.open("trajectory.gsd")

In [ ]:
len(traj)

In [ ]:
# Rendering helpers (fresnel) — supports render() and render_movie()

import io
import os
import math
import warnings

import fresnel
import IPython
import numpy as np
import packaging.version
import PIL

device = fresnel.Device()
tracer = fresnel.tracer.Path(device=device, w=300, h=300)

FRESNEL_MIN_VERSION = packaging.version.parse("0.13.0")
FRESNEL_MAX_VERSION = packaging.version.parse("0.14.0")


def _render_array(snapshot, particles=None, is_solid=None, samples=2000):
    """Return an (H, W, 4) uint8 RGBA numpy array from fresnel."""
    if (
        "version" not in dir(fresnel)
        or packaging.version.parse(fresnel.version.version) < FRESNEL_MIN_VERSION
        or packaging.version.parse(fresnel.version.version) >= FRESNEL_MAX_VERSION
    ):
        warnings.warn(f"Unsupported fresnel version {getattr(fresnel.version,'version','unknown')} - expect errors.")

    # Little "octahedron-ish" convex poly, same as tutorial helper cells
    vertices = [
        (-0.5, 0, 0),
        (0.5, 0, 0),
        (0, -0.5, 0),
        (0, 0.5, 0),
        (0, 0, -0.5),
        (0, 0, 0.5),
    ]
    poly_info = fresnel.util.convex_polyhedron_from_vertices(vertices)

    N = snapshot.particles.N
    if particles is not None:
        N = len(particles)
    if is_solid is not None:
        N = int(np.sum(is_solid))

    L = float(snapshot.configuration.box[0])

    scene = fresnel.Scene(device)
    geometry = fresnel.geometry.ConvexPolyhedron(scene, poly_info, N=N)
    geometry.material = fresnel.material.Material(
        color=fresnel.color.linear([0.01, 0.74, 0.26]), roughness=0.5
    )

    if particles is None and is_solid is None:
        geometry.position[:] = snapshot.particles.position[:]
        geometry.orientation[:] = snapshot.particles.orientation[:]
    elif particles is not None:
        geometry.position[:] = snapshot.particles.position[particles, :]
        geometry.orientation[:] = snapshot.particles.orientation[particles, :]
    elif is_solid is not None:
        geometry.position[:] = snapshot.particles.position[np.ix_(is_solid, [0, 1, 2])]
        geometry.orientation[:] = snapshot.particles.orientation[np.ix_(is_solid, [0, 1, 2, 3])]

    geometry.outline_width = 0.01
    fresnel.geometry.Box(scene, snapshot.configuration.box, box_radius=0.02)

    scene.lights = [
        fresnel.light.Light(direction=(0, 0, 1), color=(0.8, 0.8, 0.8), theta=math.pi),
        fresnel.light.Light(direction=(1, 1, 1), color=(1.1, 1.1, 1.1), theta=math.pi / 3),
    ]
    scene.camera = fresnel.camera.Orthographic(
        position=(L * 2, L, L * 2),
        look_at=(0, 0, 0),
        up=(0, 1, 0),
        height=L * 1.4 + 1,
    )
    scene.background_color = (1, 1, 1)
    scene.background_alpha = 1

    if "CI" in os.environ:
        samples = min(samples, 100)

    return tracer.sample(scene, samples=samples)


def render(snapshot, particles=None, is_solid=None, samples=2000):
    """Render a snapshot inline as a PNG image in the notebook."""
    a = _render_array(snapshot, particles=particles, is_solid=is_solid, samples=samples)
    # a is RGBA uint8 array
    return IPython.display.Image(PIL.Image.fromarray(a[:, :, :3])._repr_png_())


def render_movie(frames, particles=None, is_solid=None, duration=200, samples=500):
    """
    Create and display an animated GIF from a list of snapshots.
      - frames: iterable of snapshots
      - particles: optional list of particle indices
      - is_solid: optional list/array mask OR list of masks (one per frame)
      - duration: ms per frame
      - samples: fresnel samples per frame (keep modest for speed)
    """
    frames = list(frames)
    if len(frames) == 0:
        raise ValueError("frames is empty")

    # Allow is_solid to be a single mask or per-frame list
    if is_solid is None or isinstance(is_solid, (np.ndarray, list)) and (len(frames) != len(is_solid)):
        # If is_solid is a single mask (same for all frames), broadcast it
        if is_solid is None or (hasattr(is_solid, "__len__") and len(is_solid) == frames[0].particles.N):
            is_solid_per = [is_solid] * len(frames)
        else:
            is_solid_per = [None] * len(frames)
    else:
        is_solid_per = is_solid

    # Render first frame
    a0 = _render_array(frames[0], particles=particles, is_solid=is_solid_per[0], samples=samples)
    im0 = PIL.Image.fromarray(a0[:, :, :3]).convert("P", palette=PIL.Image.Palette.ADAPTIVE)

    ims = []
    for i, snap in enumerate(frames[1:]):
        a = _render_array(snap, particles=particles, is_solid=is_solid_per[i+1], samples=samples)
        im = PIL.Image.fromarray(a[:, :, :3])
        ims.append(im.quantize(palette=im0))

    # Add a blank end frame so the GIF "rests" on white
    blank = np.ones((im0.height, im0.width, 3), dtype=np.uint8) * 255
    ims.append(PIL.Image.fromarray(blank).quantize(palette=im0))

    f = io.BytesIO()
    im0.save(f, format="GIF", save_all=True, append_images=ims, duration=duration, loop=0)

    return IPython.display.Image(data=f.getvalue())

In [ ]:
render_movie(traj[0:50:5], particles=[12, 18])

In [ ]:
render_movie(traj[0::24])

In [ ]:
solid = freud.order.SolidLiquid(l=6, q_threshold=0.7, solid_threshold=6)
is_solid = []
for frame in traj:
    solid.compute(
        system=(frame.configuration.box, frame.particles.position),
        neighbors=dict(mode="nearest", num_neighbors=8),
    )
    is_solid.append(solid.num_connections > solid.solid_threshold)

In [ ]:
fig = matplotlib.figure.Figure(figsize=(5, 3.09))
ax = fig.add_subplot()
num_solid = numpy.array([numpy.sum(a) for a in is_solid])
ax.plot(num_solid)
ax.set_xlabel("frame")
ax.set_ylabel("number of particles in a solid environment")
fig

In [ ]:
start_frame = int(numpy.argmax(num_solid > 4))
end_frame = int(numpy.argmax(num_solid == numpy.max(num_solid)))
step = int((end_frame - start_frame) / 6)
render_movie(
    traj[start_frame:end_frame:step], is_solid=is_solid[start_frame:end_frame:step]
)

In [ ]:
print("done")